# 📦 Supply Chain Data Analytics - Exploratory Data Analysis
**Autor:** Hugo Arcanjo  
**Objetivo:** Auditar métricas operacionais de estoque, variabilidade de Lead Time e desvios logísticos da Indústria Alfa para suportar tomadas de decisão estratégicas.

### 1. Importação de Bibliotecas e Conexão com o Banco de Dados

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração visual dos gráficos
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# Conexão com o banco de dados SQLite gerado
conn = sqlite3.connect('../data/supply_chain_data.db')
print("Conexão com o banco de dados estabelecida com sucesso!")

### 2. Análise de Lead Time (Gargalo: Confiabilidade da Cadeia)
Como o SQLite não possui uma função nativa para cálculo de Desvio Padrão (`STDDEV`), extraímos os dados brutos de pedidos via SQL e realizamos o tratamento estatístico completo utilizando o `pandas` e o `numpy`.

In [ ]:
# Query para trazer os dados brutos de entrega por pedido e fornecedor
query_lead_time_bruto = """
SELECT 
    f.nome_fornecedor,
    (julianday(p.data_entrega_real) - julianday(p.data_pedido)) AS lead_time_real
FROM tb_pedidos_compra p
INNER JOIN tb_fornecedores f ON p.id_fornecedor = f.id_fornecedor
WHERE p.data_entrega_real IS NOT NULL;
"""

df_lt = pd.read_sql_query(query_lead_time_bruto, conn)

# Cálculo estatístico agrupado: Média e Desvio Padrão (Variabilidade)
df_lt_stats = df_lt.groupby('nome_fornecedor')['lead_time_real'].agg(['count', 'mean', 'std']).reset_index()
df_lt_stats.columns = ['Fornecedor', 'Total Pedidos', 'Lead Time Médio (Dias)', 'Desvio Padrão (Variabilidade)']
display(df_lt_stats.round(2))

In [ ]:
# Geração do Boxplot para analisar a dispersão e instabilidade das entregas
plt.figure(figsize=(12, 6))
sns.boxplot(data=df_lt, x='nome_fornecedor', y='lead_time_real', palette='Set2')
plt.axhline(y=7, color='r', linestyle='--', label='Prazo Contratual Padrão (7 dias)')
plt.title('Distribuição e Variabilidade do Lead Time Real por Fornecedor', fontsize=14, fontweight='bold')
plt.xlabel('Fornecedor', fontsize=12)
plt.ylabel('Lead Time Real (Dias)', fontsize=12)
plt.legend()
plt.show()

### 3. Gestão de Estoque (Gargalo: Classificação e Capital Imobilizado)
Nesta etapa, importamos a auditoria financeira do estoque e aplicamos o conceito de Engenharia de Produção da **Curva ABC** para classificar os SKUs baseado no valor total acumulado em estoque.

In [ ]:
# Lendo a query de auditoria de estoque
with open('../queries/inventory_audit.sql', 'r') as file:
    query_estoque = file.read()

df_estoque = pd.read_sql_query(query_estoque, conn)

# Calculando a representatividade e os percentuais acumulados
df_estoque = df_estoque.sort_values(by='valor_estoque_atual', ascending=False).reset_index(drop=True)
df_estoque['pct_total'] = df_estoque['valor_estoque_atual'] / df_estoque['valor_estoque_atual'].sum()
df_estoque['pct_acumulado'] = df_estoque['pct_total'].cumsum()

# Aplicação da regra da Curva ABC baseada no valor financeiro acumulado
def classificar_abc(row):
    if row['pct_acumulado'] <= 0.80:
        return 'A'
    elif row['pct_acumulado'] <= 0.95:
        return 'B'
    else:
        return 'C'

df_estoque['Classe_ABC'] = df_estoque.apply(classificar_abc, axis=1)

# Resumo da classificação ABC
resumo_abc = df_estoque.groupby('Classe_ABC').agg(
    qtd_skus=('sku', 'count'),
    valor_total_classe=('valor_estoque_atual', 'sum'),
    capital_excesso_total=('capital_imobilizado_excesso', 'sum')
).reset_index()
display(resumo_abc.round(2))

In [ ]:
# Identificação de SKUs da Classe C com excesso crônico de estoque (Insight 2)
classe_c_excesso = df_estoque[(df_estoque['Classe_ABC'] == 'C') & (df_estoque['excesso_pecas'] > 0)]
print(f"Total de Capital Congelado desnecessariamente em SKUs Classe C: R$ {classe_c_excesso['capital_imobilizado_excesso'].sum():,.2f}")
display(classe_c_excesso[['sku', 'categoria', 'estoque_atual', 'estoque_seguranca', 'capital_imobilizado_excesso']].head(5))

### 4. Análise de Desvios Logísticos (Gargalo: Qualidade e Custos Ocultos)
Aplicação do **Princípio de Pareto** para cruzar a frequência das falhas logísticas com o real impacto financeiro gerado para a Indústria Alfa.

In [ ]:
query_desvios = """
SELECT 
    tipo_desvio,
    COUNT(id_evento) AS frequencia,
    SUM(custo_prejuizo) AS custo_total_prejuizo
FROM tb_desvios_logisticos
GROUP BY tipo_desvio;
"""

df_desvios = pd.read_sql_query(query_desvios, conn)
df_desvios['pct_frequencia'] = (df_desvios['frequencia'] / df_desvios['frequencia'].sum()) * 100
df_desvios['pct_custo'] = (df_desvios['custo_total_prejuizo'] / df_desvios['custo_total_prejuizo'].sum()) * 100

print("--- Análise Comparativa de Impacto por Falha Logística ---")
display(df_desvios.round(2).sort_values(by='custo_total_prejuizo', ascending=False))

In [ ]:
# Gráfico comparativo entre Frequência vs Custo Financeiro Real
df_melted = df_desvios.melt(id_vars='tipo_desvio', value_vars=['pct_frequencia', 'pct_custo'], 
                            var_name='Metrica', value_name='Percentual')

plt.figure(figsize=(12, 6))
sns.barplot(data=df_melted, x='tipo_desvio', y='Percentual', hue='Metrica', palette='Blues_r')
plt.title('O Paradoxo de Pareto Logístico: Frequência do Evento vs Impacto Financeiro', fontsize=14, fontweight='bold')
plt.xlabel('Tipo de Desvio Logístico', fontsize=12)
plt.ylabel('Proporção Coletiva (%)', fontsize=12)
plt.legend(['% do Total de Ocorrências', '% do Total de Prejuízo Financeiro'])
plt.show()

In [ ]:
# Fechando a conexão com o banco
conn.close()
print("Análise concluída e conexão encerrada com segurança.")